# 03. Machine Learning Model Benchmarking & GroupKFold CV
### Industrial Predictive Maintenance Portfolio Project | NASA C-MAPSS Dataset

---

## 1. Objectives & Methodological Rigor
This notebook trains, evaluates, and rigorously compares 4 regression architectures:
1. **Ridge Regression**: Linear regularization baseline.
2. **Random Forest**: Non-linear bagging ensemble.
3. **XGBoost**: Extreme Gradient Boosting with shrinkage and depth penalization.
4. **LightGBM**: Fast histogram-based gradient boosting, optimized via **Optuna**.

### Cross-Validation Strategy:
We strictly enforce **GroupKFold (5 splits)** grouped on `engine_id`. Cycles from the same engine are never distributed between training and validation folds, eliminating temporal data leakage.

### Evaluation Metrics:
- **Root Mean Squared Error (RMSE)**
- **Mean Absolute Error (MAE)**
- **NASA Asymmetric Scoring Function** ($S = \sum (\exp(-d/13)-1)$ for early, $\exp(d/10)-1$ for late predictions).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.evaluator import nasa_score, evaluate_predictions

## 2. Benchmark Results Review
Loading cross-validation benchmark results generated by `scripts/train_and_benchmark.py`.

In [ ]:
benchmark_df = pd.read_csv(PROJECT_ROOT / "reports" / "model_comparison.csv")
benchmark_df

## 3. Visual Performance Comparison
Comparing RMSE, MAE, and the NASA Asymmetric Penalty across all models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
models = benchmark_df['Model']
palette = ['#2ecc71' if 'Tuned' in m else '#3498db' for m in models]

axes[0].barh(models, benchmark_df['CV_RMSE'], color=palette, edgecolor='black')
axes[0].set_title('Cross-Validation RMSE (Cycles)', fontweight='bold')
axes[0].set_xlabel('RMSE (Lower is Better)')

axes[1].barh(models, benchmark_df['CV_MAE'], color=palette, edgecolor='black')
axes[1].set_title('Cross-Validation MAE (Cycles)', fontweight='bold')
axes[1].set_xlabel('MAE (Lower is Better)')

axes[2].barh(models, benchmark_df['CV_NASA_Score'], color=palette, edgecolor='black')
axes[2].set_title('NASA Asymmetric Penalty', fontweight='bold')
axes[2].set_xlabel('Score (Lower is Better)')

plt.suptitle('NASA C-MAPSS FD001 Model Comparison (5-Fold GroupKFold CV)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Test Set Evaluation (100 Unseen Turbofan Engines)
The champion model was evaluated against the unseen `test_FD001.txt` dataset and ground-truth `RUL_FD001.txt` at the last observed cycle.

In [ ]:
# Display saved test evaluation curve
img_path = PROJECT_ROOT / "reports" / "figures" / "test_prediction_rul_curve.png"
if img_path.exists():
    import matplotlib.image as mpimg
    img = mpimg.imread(str(img_path))
    plt.figure(figsize=(14, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title('True vs Predicted Remaining Useful Life (100 Test Engines)', fontsize=14)
    plt.show()